In [1]:
import pandas as pd
import joblib

# =========================
# 1. Load Model
# =========================

model = joblib.load("isolation_forest.pkl")

print("Model loaded successfully!")

Model loaded successfully!


In [9]:
# =========================
# 2. Load Dataset
# =========================

df = pd.read_csv("test_data.csv")

# Same 4 features used during training
features = [
    "network_latency",
    "cpu_usage",
    "memory_usage",
    "packet_loss"
]

In [10]:
X = df[features]
X

,network_latency,cpu_usage,memory_usage,packet_loss
0,333.49,28.27,62.13,6.03
1,92.04,38.11,51.24,0.00
2,61.09,54.01,60.95,0.02
3,306.40,83.47,84.81,0.84
4,67.48,52.01,52.25,0.16
...,...,...,...,...
49995,260.24,72.13,54.17,7.05
49996,83.77,46.54,55.80,0.21
49997,50.00,36.24,29.88,0.51
49998,77.59,80.66,37.44,0.00


In [12]:
# =========================
# 3. Generate Predictions
# =========================

df["prediction"] = model.predict(X)

# Isolation Forest:
#  1  = Normal
# -1  = Anomaly

df["status"] = df["prediction"].map({
    1: "Normal",
    -1: "Anomaly"
})

# =========================
# 4. Generate Anomaly Score
# =========================

df["anomaly_score"] = model.decision_function(X)

# =========================
# 5. Count Results
# =========================

normal_count = (df["prediction"] == 1).sum()
anomaly_count = (df["prediction"] == -1).sum()

total = len(df)

print("\nEvaluation Results")
print("==============================")

print("Total records :", total)
print("Normal        :", normal_count)
print("Anomaly       :", anomaly_count)

print(
    "Anomaly %     :",
    round((anomaly_count / total) * 100, 2),
    "%"
)

# =========================
# 6. Compare Normal vs Anomaly
# =========================

print("\nFeature Comparison")
print("==============================")

comparison = df.groupby("status")[features].mean()

print(comparison)

# =========================
# 7. Show Top Suspicious Records
# =========================

print("\nTop 20 Suspicious Records")
print("==============================")

anomalies = df[df["prediction"] == -1]

top_anomalies = anomalies.sort_values(
    "anomaly_score",
    ascending=True
).head(20)

print(
    top_anomalies[
        features + ["anomaly_score", "status"]
    ].to_string(index=False)
)

# =========================
# 8. Save Evaluation Result
# =========================

output_file = "evaluation_results2.csv"

df.to_csv(output_file, index=False)

print("\nEvaluation results saved to:")
print(output_file)


Evaluation Results
Total records : 50000
Normal        : 42987
Anomaly       : 7013
Anomaly %     : 14.03 %

Feature Comparison
         network_latency  cpu_usage  memory_usage  packet_loss
status                                                        
Anomaly       269.186914  67.718239     71.258938     4.469806
Normal         72.205727  48.072985     56.054815     0.407331

Top 20 Suspicious Records
 network_latency  cpu_usage  memory_usage  packet_loss  anomaly_score  status
         1176.04      99.19         99.64        16.90      -0.276697 Anomaly
         1053.14      98.26         99.29        15.61      -0.269581 Anomaly
          952.33      97.73         98.09        17.76      -0.268074 Anomaly
         1180.57      94.27         98.44        19.72      -0.266852 Anomaly
         1016.97      95.31         96.83        20.55      -0.264704 Anomaly
         1033.69      99.63         94.89        16.48      -0.263358 Anomaly
         1025.33      99.72         95.06     